# Karta zaliczeniowa: nieparametryczne odpowiedniki ANOVA jedno- i wieloczynnikowej

**Przedmiot:** Statystyka  
**Kierunek:** Informatyka, studia magisterskie  
**Forma:** praca zaliczeniowa w Jupyter Notebook  


**Numer Indeksu Studenta**: 

## Cel
Dla przygotowanych zbiorów danych:
1. rozpoznaj plan badania,
2. uzasadnij wybór testu nieparametrycznego,
3. wykonaj test globalny,
4. wykonaj analizę post hoc,
5. podaj wielkość efektu tam, gdzie ma to sens,
6. napisz krótką interpretację wyniku.

## Zasady
- Pracujesz **na danych z pliku Excel**.
- Analizy wykonujesz **bez własnych funkcji pomocniczych** — bezpośrednio w kolejnych komórkach.
- Używaj tylko podstawowych bibliotek i funkcji pokazanych na zajęciach.
- Nie używaj `itertools`.
- W interpretacji odwołuj się zarówno do **wyniku statystycznego**, jak i do **kontekstu problemu**.
- Tam, gdzie to możliwe, wykonaj również **wizualizację** oraz sprawdź **skośność**.

## Kryteria oceny
- poprawny dobór testu: $20\%$,
- poprawność kodu i wyników: $35\%$,
- post hoc i wielkość efektu: $20\%$,
- interpretacja i opis statystyczny: $25\%$.


## Plik z danymi

Wraz z notebookiem otrzymujesz plik:

`dane_zaliczenie_nieparametryczne_anova.xlsx`

Arkusze:
- `kruskal`
- `friedman`
- `scheirer_ray_hare`
- `cochran_q`


## Import bibliotek

Użyj tylko bibliotek potrzebnych do:
- odczytu danych,
- obliczeń statystycznych,
- korekty wielokrotnych porównań,
- wykonania wykresów.


In [ ]:
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")


## Odczyt danych z pliku Excel

Wczytaj wszystkie potrzebne arkusze do osobnych obiektów:
- `kw_df`
- `friedman_wide`
- `srh_df`
- `cochran_df`


In [ ]:
plik_excel = "dane_zaliczenie_nieparametryczne_anova.xlsx"

kw_df = pd.read_excel(plik_excel, sheet_name="kruskal")
friedman_wide = pd.read_excel(plik_excel, sheet_name="friedman")
srh_df = pd.read_excel(plik_excel, sheet_name="scheirer_ray_hare")
cochran_df = pd.read_excel(plik_excel, sheet_name="cochran_q")


In [ ]:
kw_df.head(), friedman_wide.head(), srh_df.head(), cochran_df.head()


# Część A. Jednoczynnikowa ANOVA dla grup niezależnych → Kruskal–Wallis

## Kontekst
Badamy **czas wykonania zadania programistycznego** w trzech grupach:
- `A`,
- `B`,
- `C`.

## Twoje zadania
1. Opisz, dlaczego nie stosujemy klasycznej jednoczynnikowej ANOVA.
2. Zbadaj skośność w grupach.
3. Wykonaj wykres(y) porównujące grupy.
4. Wykonaj test Kruskala–Wallisa.
5. Oblicz wielkość efektu:
   $$
   \varepsilon^2 = \frac{H-k+1}{n-k}
   $$
6. Wykonaj post hoc metodą Dunna z korektą Holma.
7. Napisz krótki wniosek.


### A1. Uzasadnienie wyboru testu
W tej komórce tekstowej zapisz:
- jaki to jest plan badania,
- jaka jest skala zmiennej zależnej,
- czy grupy są zależne czy niezależne,
- dlaczego test rangowy jest uzasadniony.


**Miejsce na odpowiedź studenta**


### A2. Statystyki opisowe i skośność
Policz co najmniej:
- liczebność,
- średnią,
- medianę,
- odchylenie standardowe,
- skośność.


In [1]:
kw_skew = (
    kw_df.groupby("metoda")["czas_s"]
    .agg(
        liczebność="count",
        średnia="mean",
        mediana="median",
        odchylenie_std="std",
        skośność=lambda x: stats.skew(x, bias=False)
    )
)

kw_skew


NameError: name 'kw_df' is not defined

### A3. Wizualizacja
Wykonaj co najmniej dwa wykresy, np.:
- boxplot,
- violinplot.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# uzupełnij

plt.tight_layout()
plt.show()


### A4. Hipotezy
Zapisz $H_0$ i $H_1$ dla testu Kruskala–Wallisa.


**Miejsce na odpowiedź studenta**


### A5. Test globalny i wielkość efektu


In [ ]:

H_stat, p_value = stats.kruskal(
    kw_df.loc[kw_df["metoda"] == "A", "czas_s"],
    kw_df.loc[kw_df["metoda"] == "B", "czas_s"],
    kw_df.loc[kw_df["metoda"] == "C", "czas_s"]
)

n = len(kw_df)
k = kw_df["metoda"].nunique()
epsilon_sq = (H_stat - k + 1) / (n - k)

print(f"Statystyka H = {H_stat:.4f}")
print(f"p-value = {p_value:.4f}")
print(f"epsilon^2 = {epsilon_sq:.4f}")


### A6. Post hoc: Dunn + Holm

Wykonaj porównania par:
- `A` vs `B`,
- `A` vs `C`,
- `B` vs `C`.

Wskazówka: użyj rang, średnich rang, korekty na remisy i rozkładu normalnego.


In [ ]:

dunn_data = kw_df[["czas_s", "metoda"]].dropna().copy()
dunn_data["ranga"] = stats.rankdata(dunn_data["czas_s"], method="average")

N = len(dunn_data)
tie_counts = dunn_data["czas_s"].value_counts()
tie_correction = 1 - ((tie_counts**3 - tie_counts).sum() / (N**3 - N))

mean_ranks = dunn_data.groupby("metoda")["ranga"].mean()
group_sizes = dunn_data.groupby("metoda")["ranga"].size()
base_var = N * (N + 1) / 12

rows = []
poziomy = ["A", "B", "C"]

for i in range(len(poziomy) - 1):
    for j in range(i + 1, len(poziomy)):
        g1 = poziomy[i]
        g2 = poziomy[j]

        # uzupełnij:
        # diff =
        # se =
        # z_stat =
        # p_uncorrected =

        rows.append((g1, g2, z_stat, p_uncorrected))

kw_posthoc = pd.DataFrame(rows, columns=["grupa_1", "grupa_2", "z", "p_uncorrected"])
kw_posthoc["p_corrected"] = multipletests(kw_posthoc["p_uncorrected"], method="holm")[1]
kw_posthoc.sort_values("p_corrected").reset_index(drop=True)


### A7. Wniosek
Napisz w $4$–$6$ zdaniach:
- czy wynik globalny jest istotny,
- jaka jest siła efektu,
- które grupy różnią się między sobą,
- co to oznacza w kontekście problemu.


**Miejsce na odpowiedź studenta**


# Część B. Jednoczynnikowa ANOVA z powtarzanym pomiarem → Friedman

## Kontekst
Ci sami studenci rozwiązują podobne zadanie w trzech środowiskach:
- `IDE_A`,
- `IDE_B`,
- `IDE_C`.

## Twoje zadania
1. Uzasadnij wybór testu Friedmana.
2. Zbadaj skośność wyników w warunkach.
3. Wykonaj wykres pudełkowy oraz wykres profili.
4. Wykonaj test Friedmana.
5. Oblicz współczynnik zgodności Kendalla:
   $$
   W = \frac{\chi^2_F}{n(k-1)}
   $$
6. Wykonaj post hoc testami Wilcoxona dla prób zależnych z korektą Holma.
7. Napisz wniosek.


In [ ]:
friedman_long = friedman_wide.melt(
    id_vars="student_id",
    var_name="środowisko",
    value_name="czas_s"
)

friedman_long.head()


### B1. Uzasadnienie wyboru testu


**Miejsce na odpowiedź studenta**


### B2. Statystyki opisowe i skośność


In [ ]:

friedman_skew = (
    friedman_long.groupby("środowisko")["czas_s"]
    .agg(
        liczebność="count",
        średnia="mean",
        mediana="median",
        odchylenie_std="std",
        skośność=lambda x: stats.skew(x, bias=False)
    )
)

friedman_skew


### B3. Wizualizacje
Wykonaj:
- boxplot dla warunków,
- wykres profili dla kolejnych studentów.


In [ ]:
#  boxplot
plt.figure(figsize=(9, 5))

plt.show()


In [ ]:
#  wykres profili
plt.figure(figsize=(9, 5))

plt.title("Profile wyników dla tych samych studentów")
plt.ylabel("czas [s]")
plt.show()


### B4. Hipotezy
Zapisz $H_0$ i $H_1$ dla testu Friedmana.


**Miejsce na odpowiedź studenta**


### B5. Test globalny i wielkość efektu


In [ ]:

friedman_stat, friedman_p = stats.friedmanchisquare(
    friedman_wide["IDE_A"],
    friedman_wide["IDE_B"],
    friedman_wide["IDE_C"]
)

n_subjects = len(friedman_wide)
k_conditions = 3
kendalls_w = friedman_stat / (n_subjects * (k_conditions - 1))

print(f"Chi^2 Friedmana = {friedman_stat:.4f}")
print(f"p-value = {friedman_p:.4f}")
print(f"Kendall W = {kendalls_w:.4f}")


### B6. Post hoc: Wilcoxon dla prób zależnych + Holm


In [ ]:

rows = []
warunki = ["IDE_A", "IDE_B", "IDE_C"]

for i in range(len(warunki) - 1):
    for j in range(i + 1, len(warunki)):
        c1 = warunki[i]
        c2 = warunki[j]

        subset = friedman_wide[[c1, c2]].dropna()
        stat, p_uncorrected = stats.wilcoxon(subset[c1], subset[c2], zero_method="wilcox")

        rows.append((c1, c2, stat, p_uncorrected))

friedman_posthoc = pd.DataFrame(
    rows,
    columns=["warunek_1", "warunek_2", "statystyka", "p_uncorrected"]
)
friedman_posthoc["p_corrected"] = multipletests(friedman_posthoc["p_uncorrected"], method="holm")[1]
friedman_posthoc.sort_values("p_corrected").reset_index(drop=True)


### B7. Wniosek
Napisz krótki raport:
- czy warunki różnią się globalnie,
- jaka jest siła efektu,
- które pary różnią się po korekcie,
- dlaczego w tym przykładzie nie wolno użyć Kruskala–Wallisa.


**Miejsce na odpowiedź studenta**


# Część C. Dwuczynnikowa ANOVA dla grup niezależnych → Scheirer–Ray–Hare

## Kontekst
Badamy opóźnienie systemu `latency_ms` w zależności od dwóch czynników:
- `storage`: `HDD`, `SATA_SSD`, `NVMe`,
- `load`: `Niskie`, `Średnie`, `Wysokie`.

## Twoje zadania
1. Rozpoznaj plan badania.
2. Uzasadnij analizę nieparametryczną.
3. Zbadaj skośność w komórkach planu.
4. Wykonaj wizualizacje dla układu $3 \times 3$.
5. Oblicz test Scheirera–Raya–Hare'a:
   - efekt `storage`,
   - efekt `load`,
   - interakcję `storage:load`.
6. Wykonaj post hoc jako:
   - efekty proste `storage` w obrębie poziomów `load`,
   - efekty proste `load` w obrębie poziomów `storage`,
   - porównania parowe z korektą Holma.
7. Zinterpretuj wyniki.


### C1. Uzasadnienie wyboru analizy


**Miejsce na odpowiedź studenta**


### C2. Skośność w komórkach planu


In [ ]:
# 
srh_skew = (
    srh_df.groupby(["storage", "load"])["latency_ms"]
    .agg(
        liczebność="count",
        średnia="mean",
        mediana="median",
        odchylenie_std="std",
        skośność=lambda x: stats.skew(x, bias=False)
    )
)

srh_skew


### C3. Wizualizacje
Wykonaj co najmniej:
- boxplot z `hue`,
- wykres median / pointplot.


In [ ]:
# 
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# uzupełnij

plt.tight_layout()
plt.show()


### C4. Hipotezy
Zapisz hipotezy dla:
- efektu głównego `storage`,
- efektu głównego `load`,
- interakcji `storage:load`.


**Miejsce na odpowiedź studenta**


### C5. Test Scheirera–Raya–Hare'a

Wskazówka:
- najpierw zamień wyniki na rangi,
- policz sumy rang dla poziomów czynników i komórek,
- wyznacz:
  $$
  H_{storage}, \quad H_{load}, \quad H_{cells}, \quad H_{interaction} = H_{cells} - H_{storage} - H_{load}
  $$
- wartości $p$ oblicz z rozkładu $\chi^2$.


In [ ]:
# 
srh_data = srh_df[["latency_ms", "storage", "load"]].dropna().copy()
N = len(srh_data)

srh_data["ranga"] = stats.rankdata(srh_data["latency_ms"], method="average")

tie_counts = srh_data["latency_ms"].value_counts()
C = 1 - ((tie_counts**3 - tie_counts).sum() / (N**3 - N))
const = 12 / (N * (N + 1))

ranks_storage = srh_data.groupby("storage")["ranga"].agg(["sum", "size"])
H_storage = (const * ((ranks_storage["sum"]**2 / ranks_storage["size"]).sum()) - 3 * (N + 1)) / C


# ranks_load =
# H_load =
# ranks_cells =
# H_cells =
# H_interaction =

df_storage = srh_data["storage"].nunique() - 1
df_load = srh_data["load"].nunique() - 1
df_interaction = df_storage * df_load

srh_result = pd.DataFrame({
    "efekt": ["storage", "load", "storage:load"],
    "H": [H_storage, H_load, H_interaction],
    "df": [df_storage, df_load, df_interaction]
})
srh_result["p_value"] = stats.chi2.sf(srh_result["H"], srh_result["df"])
srh_result


### C6. Efekty proste `storage` w obrębie `load`
Najpierw wykonaj testy globalne Kruskala–Wallisa dla każdego poziomu `load`, a potem — jeśli to uzasadnione — porównania parowe.


In [ ]:

simple_effects_storage = []

for load_level, subset in srh_df.groupby("load"):
    groups = [
        subset.loc[subset["storage"] == "HDD", "latency_ms"],
        subset.loc[subset["storage"] == "SATA_SSD", "latency_ms"],
        subset.loc[subset["storage"] == "NVMe", "latency_ms"]
    ]
    H_stat, p_uncorrected = stats.kruskal(*groups)
    simple_effects_storage.append((f"storage | load={load_level}", H_stat, p_uncorrected))

simple_effects_storage = pd.DataFrame(
    simple_effects_storage,
    columns=["porównanie", "H", "p_uncorrected"]
)
simple_effects_storage["p_corrected"] = multipletests(simple_effects_storage["p_uncorrected"], method="holm")[1]
simple_effects_storage


### C7. Porównania parowe `storage` w obrębie każdego poziomu `load`


In [ ]:

rows = []
poziomy_load = ["Niskie", "Średnie", "Wysokie"]
poziomy_storage = ["HDD", "SATA_SSD", "NVMe"]

for load_level in poziomy_load:
    subset = srh_df.loc[srh_df["load"] == load_level, ["storage", "latency_ms"]].copy()
    subset["ranga"] = stats.rankdata(subset["latency_ms"], method="average")

    N_subset = len(subset)
    tie_counts = subset["latency_ms"].value_counts()
    tie_correction = 1 - ((tie_counts**3 - tie_counts).sum() / (N_subset**3 - N_subset))
    mean_ranks = subset.groupby("storage")["ranga"].mean()
    group_sizes = subset.groupby("storage")["ranga"].size()
    base_var = N_subset * (N_subset + 1) / 12

    for i in range(len(poziomy_storage) - 1):
        for j in range(i + 1, len(poziomy_storage)):
            g1 = poziomy_storage[i]
            g2 = poziomy_storage[j]

            # TODO:
            # diff =
            # se =
            # z_stat =
            # p_uncorrected =

            rows.append((load_level, g1, g2, z_stat, p_uncorrected))

storage_pairwise_posthoc = pd.DataFrame(
    rows,
    columns=["load", "storage_1", "storage_2", "z", "p_uncorrected"]
)
storage_pairwise_posthoc["p_corrected"] = multipletests(
    storage_pairwise_posthoc["p_uncorrected"],
    method="holm"
)[1]

storage_pairwise_posthoc.sort_values(["load", "p_corrected"]).reset_index(drop=True)


### C8. Efekty proste `load` w obrębie `storage`


In [ ]:

simple_effects_load = []

for storage_level, subset in srh_df.groupby("storage"):
    groups = [
        subset.loc[subset["load"] == "Niskie", "latency_ms"],
        subset.loc[subset["load"] == "Średnie", "latency_ms"],
        subset.loc[subset["load"] == "Wysokie", "latency_ms"]
    ]
    H_stat, p_uncorrected = stats.kruskal(*groups)
    simple_effects_load.append((f"load | storage={storage_level}", H_stat, p_uncorrected))

simple_effects_load = pd.DataFrame(
    simple_effects_load,
    columns=["porównanie", "H", "p_uncorrected"]
)
simple_effects_load["p_corrected"] = multipletests(simple_effects_load["p_uncorrected"], method="holm")[1]
simple_effects_load


### C9. Porównania parowe `load` w obrębie każdego poziomu `storage`


In [ ]:

rows = []
poziomy_storage = ["HDD", "SATA_SSD", "NVMe"]
poziomy_load = ["Niskie", "Średnie", "Wysokie"]

for storage_level in poziomy_storage:
    subset = srh_df.loc[srh_df["storage"] == storage_level, ["load", "latency_ms"]].copy()
    subset["ranga"] = stats.rankdata(subset["latency_ms"], method="average")

    N_subset = len(subset)
    tie_counts = subset["latency_ms"].value_counts()
    tie_correction = 1 - ((tie_counts**3 - tie_counts).sum() / (N_subset**3 - N_subset))
    mean_ranks = subset.groupby("load")["ranga"].mean()
    group_sizes = subset.groupby("load")["ranga"].size()
    base_var = N_subset * (N_subset + 1) / 12

    for i in range(len(poziomy_load) - 1):
        for j in range(i + 1, len(poziomy_load)):
            g1 = poziomy_load[i]
            g2 = poziomy_load[j]

            
            # diff =
            # se =
            # z_stat =
            # p_uncorrected =

            rows.append((storage_level, g1, g2, z_stat, p_uncorrected))

load_pairwise_posthoc = pd.DataFrame(
    rows,
    columns=["storage", "load_1", "load_2", "z", "p_uncorrected"]
)
load_pairwise_posthoc["p_corrected"] = multipletests(
    load_pairwise_posthoc["p_uncorrected"],
    method="holm"
)[1]

load_pairwise_posthoc.sort_values(["storage", "p_corrected"]).reset_index(drop=True)


### C10. Wniosek
W interpretacji odpowiedz:
- które efekty główne są istotne,
- czy interakcja jest istotna,
- czy należy patrzeć na efekty proste,
- jak zmienia się opóźnienie systemu w zależności od typu pamięci i obciążenia.


**Miejsce na odpowiedź studenta**


# Część D. Dane binarne $0/1$ w pomiarach zależnych → Cochran $Q$

## Kontekst
Dla tych samych obiektów sprawdzamy, czy trzy algorytmy klasyfikacyjne:
- `alg_A`,
- `alg_B`,
- `alg_C`

poprawnie wykrywają obiekt ($1$ = sukces, $0$ = błąd).

## Twoje zadania
1. Wyjaśnij, dlaczego nie badamy tu skośności.
2. Podaj, dlaczego używamy testu Cochran $Q$.
3. Policz odsetki sukcesów.
4. Wykonaj test globalny.
5. Wykonaj post hoc testami McNemary z korektą Holma.
6. Napisz wniosek.


### D1. Uzasadnienie wyboru testu


**Miejsce na odpowiedź studenta**


### D2. Statystyki opisowe


In [ ]:

cochran_df.mean().rename("odsetek sukcesów")


### D3. Test globalny


In [ ]:

cochran_result = cochrans_q(cochran_df)
cochran_result


### D4. Post hoc: McNemar + Holm


In [ ]:

rows = []
algorytmy = ["alg_A", "alg_B", "alg_C"]

for i in range(len(algorytmy) - 1):
    for j in range(i + 1, len(algorytmy)):
        c1 = algorytmy[i]
        c2 = algorytmy[j]

        table = pd.crosstab(cochran_df[c1], cochran_df[c2]).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
        result = mcnemar(table, exact=True)

        rows.append((c1, c2, result.statistic, result.pvalue, int(table.loc[0, 1]), int(table.loc[1, 0])))

cochran_posthoc = pd.DataFrame(
    rows,
    columns=["warunek_1", "warunek_2", "statystyka", "p_uncorrected", "n_01", "n_10"]
)
cochran_posthoc["p_corrected"] = multipletests(cochran_posthoc["p_uncorrected"], method="holm")[1]
cochran_posthoc.sort_values("p_corrected").reset_index(drop=True)


### D5. Wniosek
Napisz:
- czy algorytmy różnią się globalnie,
- które pary różnią się po korekcie,
- czy istotny wynik globalny oznacza automatycznie istotność każdej pary.


**Miejsce na odpowiedź studenta**


# Raport końcowy

Na końcu notebooka przygotuj krótkie podsumowanie całej pracy.

## Minimalna zawartość raportu
Dla każdej części:
1. nazwa testu,
2. hipotezy,
3. statystyka testowa i $p$,
4. wielkość efektu — jeśli dotyczy,
5. wynik post hoc,
6. jednoznaczny wniosek merytoryczny.

## Pytania kontrolne
1. Dlaczego Kruskal–Wallis nie nadaje się do powtarzanego pomiaru?
2. Czym różni się plan badania dla Friedmana i Cochran $Q$?
3. Kiedy w analizie dwuczynnikowej ważniejsza od efektów głównych jest interakcja?
4. Dlaczego w części z danymi binarnymi nie analizujemy skośności?


**Miejsce na końcowe podsumowanie studenta**
